# Fine-Tuning AI Search for E-commerce: Workshop Lab

**Welcome.** Over the next 45 minutes you'll compare BM25, generic dense retrieval, and a fine-tuned SPLADE model on real Amazon ESCI data (queries graded Exact, Substitute, Complement, and Irrelevant), then fold in hybrid search as a production pattern.

### What You'll Do

1. **CP1: BM25 baseline (12 min).** The 10 demo queries against the lexical baseline.
2. **CP2: BM25 vs generic dense vs fine-tuned SPLADE (20 min).** First ranking-quality metrics.
3. **CP3: Production hybrid fusion (13 min).** Fuse dense + SPLADE with DBSF via Qdrant's Query API.
4. **Wrap (15 min).** Full 2K-query eval, query-level findings, takeaways.

> **The corpus is a 35-40K-product ESCI subset, not the full Amazon catalog.** We chose it from the relevance labels so every graded product is reachable at eval time; queries for products outside it return the closest match, not nothing. In production you'd index the full catalog first, then evaluate against query logs, judgments, or clicks.

## Setup

One cell: connect to Qdrant, load the ESCI labels for our eval queries, attach labels to the demo queries, and start the encoders we'll reuse throughout the lab.

In [ ]:
from pathlib import Path
import json
import sys

# Cursor/Jupyter may start the kernel in the repo root or in notebooks/.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
DATA = REPO_ROOT / "data"

import pandas as pd
from datasets import load_dataset
from IPython.display import display, Markdown
from qdrant_client import QdrantClient, models
from tqdm import tqdm

from eval.metrics import ndcg_at_k, mrr_at_k, recall_at_k, precision_at_k
from eval.viewer import compare_results, inspect_sparse_vector
from retrieval import SpladeEncoder
from scripts.setup_collections import (
    COLLECTION,
    DENSE_VECTOR_NAME,
    BM25_VECTOR_NAME,
    SPLADE_VECTOR_NAME,
    BM25_MODEL as BM25_MODEL_ID,
    DENSE_MODEL as DENSE_MODEL_ID,
    SPLADE_FINETUNED_MODEL_DEFAULT,
)

client = QdrantClient(host="localhost", port=6333)
points_count = client.get_collection(COLLECTION).points_count or 0

splade_encoder = SpladeEncoder(SPLADE_FINETUNED_MODEL_DEFAULT, device="cpu")

# The manifest tells the notebook which 2,000 ESCI queries we evaluate.
manifest = json.loads((DATA / "corpus_manifest.json").read_text())
eval_qids = set(manifest["eval_query_ids"])

demo_queries = json.loads((DATA / "demo_queries.json").read_text())
print(f"Loading labels for {len(eval_qids):,} ESCI eval queries...")
esci_rows = load_dataset(manifest["dataset_id"], split=manifest["split"])


def normalize_grade(label):
    """Keep only the ESCI grade letter: E, S, C, or I."""
    return str(label)[0].upper() if label else "I"


queries_by_id = {}
for row in esci_rows:
    if str(row.get("product_locale", "us")).lower() != manifest["locale"]:
        continue

    qid = int(row["query_id"])
    if qid not in eval_qids:
        continue

    if qid not in queries_by_id:
        queries_by_id[qid] = {
            "query_id": qid,
            "query": row["query"],
            "qrels": {},
        }

    product_id = row["product_id"]
    queries_by_id[qid]["qrels"][product_id] = normalize_grade(row["esci_label"])

# qrels are the relevance labels: product_id -> E/S/C/I for one query.
full_2k = [queries_by_id[qid] for qid in sorted(eval_qids)]
for query in demo_queries:
    query["qrels"] = queries_by_id[int(query["esci_qid"])]["qrels"]

display(Markdown(
    f"**Ready.** Corpus: {points_count:,} products · "
    f"{len(demo_queries)} demo queries · {len(full_2k):,} eval queries."
))


def retrieved_ids(points):
    """Return ESCI product IDs from Qdrant results."""
    return [p.payload.get("product_id", p.id) for p in points]


---
### How the Collection Was Created

Before querying the prebuilt 35-40K-product `products` collection, this cell shows how it was created: same API surface, scaled to six products so the schema is easy to read. The named vectors we register here (one dense, two sparse) are exactly what `scripts/setup_collections.py` registers at full scale.

In [ ]:
# Embedded Qdrant inside the kernel -- no second server, no network.
mini_client = QdrantClient(":memory:")

mini_products = [
    {"product_id": "demo-1", "product_title": "Sony 65 inch 4K HDR Smart TV"},
    {"product_id": "demo-2", "product_title": "LG 55 inch OLED Smart TV"},
    {"product_id": "demo-3", "product_title": "Apple iPhone 13 silicone case, Midnight"},
    {"product_id": "demo-4", "product_title": "Apple iPhone 11 leather case, Saddle Brown"},
    {"product_id": "demo-5", "product_title": "10 gallon glass fish aquarium tank"},
    {"product_id": "demo-6", "product_title": "5 gallon plastic storage bucket"},
]

# The real collection stores three named vectors per product.
# BM25 uses IDF, so rare words count more. SPLADE already has learned weights.
mini_client.create_collection(
    collection_name="mini_products",
    vectors_config={
        DENSE_VECTOR_NAME: models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        BM25_VECTOR_NAME: models.SparseVectorParams(modifier=models.Modifier.IDF),
        SPLADE_VECTOR_NAME: models.SparseVectorParams(),
    },
)

# SPLADE is a custom model, so we encode the titles ourselves (one batched call).
splade_vectors = splade_encoder.encode([p["product_title"] for p in mini_products])


def make_point(point_id, product, splade_encoded):
    """One product -> one Qdrant point: three named vectors plus the product as payload."""
    title = product["product_title"]
    splade_indices, splade_values = splade_encoded
    return models.PointStruct(
        id=point_id,
        vector={
            # Dense and BM25: qdrant-client knows these models and encodes the title for us.
            DENSE_VECTOR_NAME: models.Document(text=title, model=DENSE_MODEL_ID),
            BM25_VECTOR_NAME: models.Document(text=title, model=BM25_MODEL_ID),
            # SPLADE: custom model, so we hand over the sparse vector we encoded above.
            SPLADE_VECTOR_NAME: models.SparseVector(
                indices=list(map(int, splade_indices)),
                values=list(map(float, splade_values)),
            ),
        },
        payload=product,
    )


# Each product becomes one point. Build them all, then upsert in a single call.
points = [make_point(i, product, splade_vectors[i]) for i, product in enumerate(mini_products)]

mini_client.upsert(collection_name="mini_products", points=points)
print(f"Created mini_products with {len(points)} points")


In [ ]:
mini_query = "65 inch tv"
mini_results = mini_client.query_points(
    collection_name="mini_products",
    query=models.Document(text=mini_query, model=DENSE_MODEL_ID),
    using=DENSE_VECTOR_NAME,
    limit=3,
    with_payload=True,
).points

print(f"query: {mini_query!r}")
for hit in mini_results:
    print(f"  {hit.score:.3f}  {hit.payload['product_title']}")


### Open the Qdrant Dashboard

The mini collection above runs inside this notebook kernel, so it does not appear in the dashboard. The dashboard shows the real local `products` collection that the rest of the lab queries.

Open the Qdrant dashboard: http://localhost:6333/dashboard

In `products`, look for:

- Point count around 35K-40K products
- Named vectors: `dense`, `bm25`, and `splade_finetuned`
- Payload fields such as `product_id`, `product_title`, `product_brand`, and `product_color`

Dense vectors are not readable by eye. Sparse vectors are token IDs plus weights; later, the SPLADE inspection cell maps those token IDs back to readable terms.

---
### Shared Retrievers

Four retrievers, one set, used by every checkpoint that follows. Each takes a query text, encodes it in the appropriate format, and calls Qdrant.

In [ ]:
def search_dense(query, k=10):
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=query, model=DENSE_MODEL_ID),
        using=DENSE_VECTOR_NAME,
        limit=k,
        with_payload=True,
    ).points


def search_bm25(query, k=10):
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=query, model=BM25_MODEL_ID),
        using=BM25_VECTOR_NAME,
        limit=k,
        with_payload=True,
    ).points


def splade_query_vector(query):
    """Encode query text with SPLADE and return a Qdrant sparse vector."""
    indices, values = splade_encoder.encode([query])[0]
    return models.SparseVector(
        indices=list(map(int, indices)),
        values=list(map(float, values)),
    )


def search_splade(query, k=10):
    return client.query_points(
        collection_name=COLLECTION,
        query=splade_query_vector(query),
        using=SPLADE_VECTOR_NAME,
        limit=k,
        with_payload=True,
    ).points


def search_hybrid_splade(query, k=10, candidates=100):
    generic_dense = models.Prefetch(
        query=models.Document(text=query, model=DENSE_MODEL_ID),
        using=DENSE_VECTOR_NAME,
        limit=candidates,
    )
    splade = models.Prefetch(
        query=splade_query_vector(query),
        using=SPLADE_VECTOR_NAME,
        limit=candidates,
    )

    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[generic_dense, splade],
        query=models.FusionQuery(fusion=models.Fusion.DBSF),
        limit=k,
        with_payload=True,
    ).points


RETRIEVERS = {
    "BM25": search_bm25,
    "Generic dense": search_dense,
    "SPLADE": search_splade,
    "Hybrid DBSF": search_hybrid_splade,
}


---
## CP1: BM25 Baseline

**Goal:** start from the traditional lexical baseline for product search.

**No aggregate metric yet.** Each row is tagged with its ESCI grade (E/S/C/I). BM25 is strong when the shopper's words match the product title, but it has no learned understanding of paraphrases or product relationships.

In [ ]:
# Run BM25 across all 10 demo queries; cache for CP2 side-by-side.
bm25_results = {q["id"]: search_bm25(q["text"]) for q in demo_queries}

# Show three focus demo queries in full,
# then summarize the other seven in a compact table.
#   - specificity (size+brand)  -> ele_65_lg_tv (matches the slide hook)
#   - identity (brand+model)    -> ele_apple_iphone_11_case
#   - specificity (quantity)    -> hom_11_piece_knife_set_without_block
focus_ids = [
    "ele_65_lg_tv",
    "ele_apple_iphone_11_case",
    "hom_11_piece_knife_set_without_block",
]

for qid in focus_ids:
    q = next(h for h in demo_queries if h["id"] == qid)
    display(Markdown(f"### `{q['id']}` — *{q['category']}* — **{q['text']}**"))
    compare_results(
        query=q["text"],
        models_results={"BM25 baseline": bm25_results[qid]},
        esci_qrels=q["qrels"],
    )

# Compact summary for the other seven demo queries.
_rows = []
for h in demo_queries:
    if h["id"] in focus_ids:
        continue
    pts = bm25_results[h["id"]]
    top = pts[0] if pts else None
    top_pid = top.payload.get("product_id", top.id) if top else None
    top_title = top.payload.get("product_title", top_pid) if top else "—"
    grade = h["qrels"].get(top_pid, "—") if top_pid else "—"
    _rows.append({
        "query_id": h["id"],
        "query_text": h["text"],
        "rank-1 title": str(top_title),
        "rank-1 grade": grade,
    })
display(Markdown("#### Remaining seven demo queries — rank-1 snapshot"))
display(pd.DataFrame(_rows).style.hide(axis="index"))


---
## CP2: BM25, Generic Dense, and Fine-Tuned SPLADE

**Goal:** compare the traditional lexical baseline, a generic dense model, and the fine-tuned SPLADE model on the same queries.

Same demo queries. Three retrievers side-by-side on the focus demo queries. Then the first aggregate numbers across the demo set: illustrative only; the headline 2K claim lands in the wrap.

**Read the metrics in product-search terms:**

- **nDCG@10:** did the best judged products show up near the top?
- **MRR@10:** how soon did the first clearly good product appear?
- **Recall@10:** how many of the known good products did the top 10 recover?
- **Precision@10:** how much of the top 10 was known good?

In [ ]:
dense_results = {q["id"]: search_dense(q["text"]) for q in demo_queries}
splade_results = {q["id"]: search_splade(q["text"]) for q in demo_queries}

# Side-by-side on two demo queries where the lift is most legible.
cp2_focus = ["ele_65_lg_tv", "hom_11_piece_knife_set_without_block"]
for qid in cp2_focus:
    q = next(h for h in demo_queries if h["id"] == qid)
    display(Markdown(f"### `{q['id']}` — **{q['text']}**"))
    compare_results(
        query=q["text"],
        models_results={
            "BM25 baseline": bm25_results[qid],
            "Generic dense": dense_results[qid],
            "Fine-tuned SPLADE": splade_results[qid],
        },
        esci_qrels=q["qrels"],
    )

# First metric reveal: demo-mean metrics across BM25 + generic dense + SPLADE.
# Illustrative only -- n=10 is too small for stable numbers. Headline 2K claim in wrap.
def demo_metrics(retrievers):
    per_q = {name: {"ndcg": [], "mrr": [], "recall": [], "precision": []} for name in retrievers}
    for h in demo_queries:
        for name, fn in retrievers.items():
            ids = retrieved_ids(fn(h["text"]))
            per_q[name]["ndcg"].append(ndcg_at_k(ids, h["qrels"], k=10))
            per_q[name]["mrr"].append(mrr_at_k(ids, h["qrels"], k=10))
            per_q[name]["recall"].append(recall_at_k(ids, h["qrels"], k=10))
            per_q[name]["precision"].append(precision_at_k(ids, h["qrels"], k=10))
    return per_q

per_q_3way = demo_metrics({
    "BM25":   search_bm25,
    "Generic dense":  search_dense,
    "SPLADE": search_splade,
})

_rows = []
for name, scores in per_q_3way.items():
    _rows.append({
        "approach": name,
        "nDCG@10":  sum(scores["ndcg"])  / len(scores["ndcg"]),
        "MRR@10":   sum(scores["mrr"])   / len(scores["mrr"]),
        "Recall@10": sum(scores["recall"]) / len(scores["recall"]),
        "Precision@10": sum(scores["precision"]) / len(scores["precision"]),
    })
_df = pd.DataFrame(_rows)
display(Markdown("#### Demo-set metrics (n=10) — *illustrative, not the headline claim*"))
display(_df.style.hide(axis="index").format({
    "nDCG@10": "{:.3f}",
    "MRR@10": "{:.3f}",
    "Recall@10": "{:.3f}",
    "Precision@10": "{:.3f}",
}))


### Peek Inside a Sparse Vector (2 min)

Sparse is **interpretable** in a way dense isn't. The next cell encodes the query `"10 gallon fish tank"` with fine-tuned SPLADE and shows the top-weighted tokens. You should see the obvious literals (`tank`, `gallon`, `fish`, `10`), plus the **learned term expansions**: tokens like `tanks`, `capacity`, and `aquarium` that weren't typed but appear because fine-tuning taught the model they're semantically related to fish-tank product titles. That's the lift over raw lexical matching.

*(The viewer filters punctuation and stopwords by default; pass `skip_uninformative=False` to see the raw top-K.)*

In [ ]:
vocab = json.loads((DATA / "splade_vocab.json").read_text())
vocab = {int(k): v for k, v in vocab.items()}  # JSON keys come in as strings

q_idx, q_vals = splade_encoder.encode(["10 gallon fish tank"])[0]
_ = inspect_sparse_vector((q_idx, q_vals), vocab=vocab, top_k=12)


---
## CP3: Production Hybrid Fusion

**Goal:** show the production pattern. Combine generic dense with fine-tuned SPLADE using **Distribution-Based Score Fusion** in a single Qdrant Query API call.

The API shape: two `Prefetch` blocks (one dense, one SPLADE), each over-fetching candidates, then `FusionQuery(fusion=Fusion.DBSF)` returns the final top 10. DBSF normalizes each retriever's scores before adding them, which beats rank-only fusion here because dense and SPLADE both carry useful score strength.

If the 10-query demo set shows hybrid below SPLADE, don't over-read it: the demo set is for inspection, the full 2K eval is the stable comparison.

In [ ]:
hybrid_splade_results = {q["id"]: search_hybrid_splade(q["text"]) for q in demo_queries}

# Continue the slide-hook through-line: 65 lg tv lands here so attendees see
# the same query from slide 2 -> CP1 -> CP2 -> CP3.
qid = "ele_65_lg_tv"
q = next(h for h in demo_queries if h["id"] == qid)
display(Markdown(f"### Generic dense + SPLADE DBSF hybrid on `{qid}` — **{q['text']}**"))
compare_results(
    query=q["text"],
    models_results={
        "BM25 baseline": bm25_results[qid],
        "Generic dense": dense_results[qid],
        "Fine-tuned SPLADE": splade_results[qid],
        "Hybrid DBSF": hybrid_splade_results[qid],
    },
    esci_qrels=q["qrels"],
)

# 4-way demo-set refresh -- same caveat as CP2 (illustrative, not the headline).
per_q_4way = demo_metrics(RETRIEVERS)

_rows = []
for name in RETRIEVERS:
    _rows.append({
        "approach": name,
        "nDCG@10":  sum(per_q_4way[name]["ndcg"])  / len(per_q_4way[name]["ndcg"]),
        "MRR@10":   sum(per_q_4way[name]["mrr"])   / len(per_q_4way[name]["mrr"]),
        "Recall@10": sum(per_q_4way[name]["recall"]) / len(per_q_4way[name]["recall"]),
        "Precision@10": sum(per_q_4way[name]["precision"]) / len(per_q_4way[name]["precision"]),
    })
_df = pd.DataFrame(_rows)
display(Markdown("#### Demo-set 4-way metrics (n=10) — *illustrative*"))
display(_df.style.hide(axis="index").format({
    "nDCG@10": "{:.3f}",
    "MRR@10": "{:.3f}",
    "Recall@10": "{:.3f}",
    "Precision@10": "{:.3f}",
}))


---
## Wrap: Headline Numbers, Findings, and Playbook (15 min)

### The Authoritative Claim: Full 2K Eval

Computed **live** against the full 2K ESCI test split. **This is the number we stand behind.**

Read the first three rows as the model-quality comparison: BM25 baseline, generic dense retrieval, and fine-tuned SPLADE. Read the hybrid row as the production retrieval pattern, not as proof that fine-tuning helped.

We lead with **nDCG@10** because it uses the ESCI grades and rewards putting the best products higher. MRR, Recall, and Precision are supporting checks from the same run, not three separate claims.

Running the full 2K eval live takes a few minutes on a 4-vCPU machine (CPU-dependent). Each retriever encodes the query locally, queries Qdrant, and records the top-10 IDs.

In [ ]:
per_query = {name: {"ndcg": [], "mrr": [], "recall": [], "precision": []} for name in RETRIEVERS}

for item in tqdm(full_2k, desc="2K eval", unit="q"):
    q_text, q_rels = item["query"], item["qrels"]
    for name, fn in RETRIEVERS.items():
        ids = retrieved_ids(fn(q_text))
        per_query[name]["ndcg"].append(ndcg_at_k(ids, q_rels, k=10))
        per_query[name]["mrr"].append(mrr_at_k(ids, q_rels, k=10))
        per_query[name]["recall"].append(recall_at_k(ids, q_rels, k=10))
        per_query[name]["precision"].append(precision_at_k(ids, q_rels, k=10))

full_rows = []
for name, scores in per_query.items():
    full_rows.append({
        "approach": name,
        "nDCG@10": sum(scores["ndcg"]) / len(scores["ndcg"]),
        "MRR@10": sum(scores["mrr"]) / len(scores["mrr"]),
        "Recall@10": sum(scores["recall"]) / len(scores["recall"]),
        "Precision@10": sum(scores["precision"]) / len(scores["precision"]),
    })

display(Markdown(f"**Live 2K eval complete** — {len(full_2k)} queries, {len(RETRIEVERS)} retrievers."))


In [ ]:
full_df = pd.DataFrame(full_rows)

bm25_ndcg = float(full_df.loc[full_df["approach"] == "BM25", "nDCG@10"].iloc[0])
full_df["delta vs BM25"] = full_df["nDCG@10"] - bm25_ndcg
full_df["relative lift vs BM25"] = full_df["nDCG@10"] / bm25_ndcg - 1.0

display(Markdown("#### Full 2K ESCI test set — headline"))
display(full_df[[
    "approach",
    "nDCG@10",
    "delta vs BM25",
    "relative lift vs BM25",
]].style.hide(axis="index").format({
    "nDCG@10": "{:.3f}",
    "delta vs BM25": "{:+.3f}",
    "relative lift vs BM25": "{:+.1%}",
}))

display(Markdown("#### Supporting metrics — same run"))
display(full_df[[
    "approach",
    "MRR@10",
    "Recall@10",
    "Precision@10",
]].style.hide(axis="index").format({
    "MRR@10": "{:.3f}",
    "Recall@10": "{:.3f}",
    "Precision@10": "{:.3f}",
}))


### When Fine-Tuning Helps (and When It Doesn't)

The aggregate hides the variance: no retriever wins every query. Two views make the pattern concrete.

- **Fine-tuning is a targeted lever, not a blanket win.** It helps most on **numeric / size / quantity** queries and barely moves the large **short brand / item** bucket, where BM25 already matches the literal title.
- **Its biggest wins are recoveries.** The top single-query gains are misspellings and rare brands where BM25 returned nothing and the learned expansions still found the product.

nDCG@10 uses the ESCI grades (Exact = 3, Substitute = 2, Complement = 1, Irrelevant = 0), so it rewards good-but-not-exact matches. The query-shape labels are rough heuristics for discussion, not ground truth.

In [ ]:
def query_shape(text):
    """Rough labels for discussing which query types moved."""
    text = text.lower()
    tokens = text.replace("-", " ").replace("/", " ").split()

    broad = ["toys for", "gifts for", "gift for", "for women", "for men", "for kids", "for girls", "for boys", "best "]
    numeric_words = {"inch", "inches", "oz", "ounce", "ounces", "gallon", "gallons", "gb", "tb", "size", "piece", "pieces", "pack", "count", "ft", "feet", "x"}
    compatibility = ["iphone", "apple watch", "galaxy", "case", "charger", "screen protector", "replacement", "compatible"]

    if any(marker in text for marker in broad):
        return "broad recommendation"
    if any(any(ch.isdigit() for ch in token) for token in tokens) or any(word in tokens for word in numeric_words):
        return "numeric / size / quantity"
    if any(marker in text for marker in compatibility):
        return "model / compatibility"
    if len(tokens) <= 3:
        return "short brand / item"
    return "other product queries"


# Per-query nDCG: fine-tuned SPLADE vs the BM25 baseline, tagged by query shape.
lift_df = pd.DataFrame([
    {
        "query": item["query"],
        "query shape": query_shape(item["query"]),
        "BM25 nDCG@10": per_query["BM25"]["ndcg"][i],
        "SPLADE nDCG@10": per_query["SPLADE"]["ndcg"][i],
    }
    for i, item in enumerate(full_2k)
])
lift_df["SPLADE - BM25"] = lift_df["SPLADE nDCG@10"] - lift_df["BM25 nDCG@10"]

# 1. When does fine-tuning help? Average lift and win rate by query shape.
by_shape = (
    lift_df.groupby("query shape")
    .agg(
        queries=("query", "count"),
        avg_lift=("SPLADE - BM25", "mean"),
        win_rate=("SPLADE - BM25", lambda s: (s > 0).mean()),
    )
    .reset_index()
    .sort_values("avg_lift", ascending=False)
    .rename(columns={"avg_lift": "avg SPLADE - BM25", "win_rate": "SPLADE wins vs BM25"})
)
display(Markdown("#### When does fine-tuning help? Lift by query shape"))
display(by_shape.style.hide(axis="index").format({
    "avg SPLADE - BM25": "{:+.3f}",
    "SPLADE wins vs BM25": "{:.0%}",
}))

# 2. What does it win? The clearest single-query gains over BM25.
display(Markdown("#### Biggest single-query wins over BM25"))
display(lift_df.sort_values("SPLADE - BM25", ascending=False).head(6)[[
    "query", "query shape", "BM25 nDCG@10", "SPLADE nDCG@10", "SPLADE - BM25"
]].style.hide(axis="index").format({
    "BM25 nDCG@10": "{:.3f}",
    "SPLADE nDCG@10": "{:.3f}",
    "SPLADE - BM25": "{:+.3f}",
}))


### Apply This to Your Own System

Use the same loop on your own catalog:

1. Start with a BM25 baseline and a real eval set.
2. Add dense (HNSW-indexed) retrieval for semantic candidates at scale.
3. Inspect failures by query shape: attributes, compatibility, broad intent, spelling, brand and model variants.
4. Fine-tune when failures cluster around domain-specific language or product constraints.
5. Hybridize signals only after each signal earns its place on held-out queries.
6. Add a reranker when the candidate set is good but the final top-10 order still needs help.
7. Route non-shopping or off-catalog intent before retrieval; a product retriever will always return products from its product index.

Next notebook: `notebooks/splade_training.ipynb` shows the fine-tuning side of that loop: labels, hard negatives, training, and evaluation. Start there when you want to train the SPLADE model on your own catalog instead of only querying this prebuilt one.

---
### Takeaway Links

- **Article series:** [Sparse Embeddings for E-commerce, Part 1](https://qdrant.tech/articles/sparse-embeddings-ecommerce-part-1/)
- **Fine-tuned model:** [thierrydamiba/splade-ecommerce-esci on Hugging Face](https://huggingface.co/thierrydamiba/splade-ecommerce-esci)
- **Amazon ESCI dataset:** [amazon-science/esci-data on GitHub](https://github.com/amazon-science/esci-data)
- **Collection setup (replicate elsewhere):** `scripts/setup_collections.py`
- **Training notebook:** `notebooks/splade_training.ipynb`

Take the lab pattern back to your own catalog: measure, inspect, fine-tune, evaluate, then decide how to serve it.